# Day 3 — Robust Code: Exceptions, Logging & Retries

**Module 2 · Python for AI Testing & Automation**

---

## What we'll cover

| # | Topic | Why it matters |
|---|---|---|
| 1 | Exceptions — mechanics | LLM APIs fail — you need to catch and handle failures |
| 2 | Exception hierarchy | Know exactly which class to catch |
| 3 | `try / except / else / finally` | Full error-handling grammar |
| 4 | Logging vs `print` | Tests leave logs, not print statements |
| 5 | Log levels and configuration | Control what you see and when |
| 6a | **Decorators** — from scratch to factory | Building block of the retry pattern |
| 6b | **Generators** — `yield`, lazy evaluation, streaming | Why LLM streaming works the way it does |
| 6c | **Retry with exponential backoff** | Putting decorators + iteration together |
| 7 | Putting it together | A production-grade LLM client |

**Why today matters:** a test framework that crashes on the first API error is useless in practice. LLMs are flaky by nature — rate limits, 500s, network blips, slow responses. Robustness is not optional. And the tools we use to build robustness (decorators, generators) are Python fundamentals that appear everywhere — pytest fixtures, streaming APIs, context managers.

---

---
## 1. Exceptions — The Mechanics

> **Plain English:** exceptions are the "oh crap" moments in code. Your program hits something it can't handle — a file that doesn't exist, a network that dropped, a JSON that won't parse. Two choices: prepare for it (`try/except`) or let the program crash. For production test code, always prepare.

In [14]:
number1 = int(input("Enter the first number: "))
number2 = int(input("Enter the second number: "))
try:
    result = number1 / number2
    print(f"The result of dividing {number1} by {number2} is: {result}")
except ZeroDivisionError as e:
    result = number1 / 1  # Default to dividing by 1 if the second number is zero
    print(f"Cannot divide by zero. Defaulting to dividing by 1. The result is: {result}")

The result of dividing 10 by 1 is: 10.0


In [15]:
# The crash — no handling
try:
    result = 10 / 0
except :
    pass

# Without the try/except this would terminate the notebook kernel
print("Execution continues after the handled exception")

Execution continues after the handled exception


In [4]:
# The anti-pattern — NEVER do this
# This catches EVERYTHING, including KeyboardInterrupt, SystemExit, and bugs you want to see

try:
    x = 10 / 0
except:           # bare except — DO NOT USE
    pass          # silently swallows errors — impossible to debug

# Why it's dangerous: if your API key is wrong (AuthenticationError),
# a bare except will hide it and you'll spend an hour wondering why nothing works.
print("Rule: always catch the SPECIFIC exception class you expect.")

Rule: always catch the SPECIFIC exception class you expect.


In [ ]:
import json
# Catching multiple specific exceptions
def risky_parse(text: str) -> dict:
    import json
    # Could raise: json.JSONDecodeError, ValueError, or KeyError on access
    data = json.loads(text)    # JSONDecodeError if invalid JSON
    return {
        "value": int(data["value"]),   # ValueError if not a number; KeyError if missing
    }

test_inputs = [
    '{"value": "42"}',          # valid — value is a string of a number
    '{"value": "not-a-number"}', # ValueError
    '{"other": 1}',             # KeyError — 'value' missing
    'not json at all',           # JSONDecodeError
]

for input in test_inputs:
    print(f"Testing input: {input}")
    try:
        result = risky_parse(input)
        print(f"Parsed result: {result}")
    except (KeyError,ValueError) as e:
        print(f"Key/value error: {e}")
    except json.JSONDecodeError as e:
        print(f"JSON decode error: {e}")
    # except json.JSONDecodeError as e:
    #     print(f"JSON decode error: {e}")
# for inp in test_inputs:
#     try:
#         result = risky_parse(inp)
#         print(f"  OK: {result}")
#     except json.JSONDecodeError as e:
#         print(f"  JSONDecodeError: {e}")
#     except (KeyError, ValueError) as e:
#         print(f"  Data error ({type(e).__name__}): {e}")

Testing input: {"value": "42"}
Parsed result: {'value': 42}
Testing input: {"value": "not-a-number"}
Key error: invalid literal for int() with base 10: 'not-a-number'
Testing input: {"other": 1}
Key error: 'value'
Testing input: not json at all
Key error: Expecting value: line 1 column 1 (char 0)


---
## 2. The Exception Hierarchy

All Python exceptions inherit from `BaseException`. Most you'll use inherit from `Exception`. Knowing the tree tells you which classes to catch.

```
BaseException
├── SystemExit            ← sys.exit() — don't catch
├── KeyboardInterrupt     ← Ctrl+C — don't catch (usually)
└── Exception             ← catch here and below
    ├── ValueError        ← wrong value: int("abc")
    ├── TypeError         ← wrong type: "a" + 1
    ├── KeyError          ← missing dict key: d["missing"]
    ├── IndexError        ← list out of range: [1,2][99]
    ├── AttributeError    ← missing attribute: None.strip()
    ├── FileNotFoundError ← open("missing.txt")
    ├── json.JSONDecodeError  ← json.loads("bad")
    ├── OSError / IOError    ← file system, network
    └── RuntimeError         ← generic runtime problem
```

Third-party libraries define their own subclasses (e.g., `openai.RateLimitError`).

In [23]:
# LLM-testing-specific exceptions you'll see constantly

# Simulate openai exceptions (structure without actually calling the API)
class MockRateLimitError(Exception): pass
class MockAPIConnectionError(Exception): pass
class MockAuthenticationError(Exception): pass
class MockAPIError(Exception): pass

exceptions_to_handle = [
    (MockRateLimitError("429 Too Many Requests"), "rate-limited — add delay and retry"),
    (MockAPIConnectionError("Connection refused"), "network issue — retry later"),
    (MockAuthenticationError("Invalid API key"), "auth failure — check .env, do NOT retry"),
    (MockAPIError("500 Internal Server Error"), "provider error — retry with backoff"),
]

for exc, guidance in exceptions_to_handle:
    print(f"  {type(exc).__name__:30s} → {guidance}")

  MockRateLimitError             → rate-limited — add delay and retry
  MockAPIConnectionError         → network issue — retry later
  MockAuthenticationError        → auth failure — check .env, do NOT retry
  MockAPIError                   → provider error — retry with backoff


---
## 3. Full `try / except / else / finally` Grammar

In [24]:
import json

def process_response(raw: str) -> dict:
    """
    Full error-handling grammar:
      try    — the code that might fail
      except — handle specific failure
      else   — runs ONLY if no exception was raised (often missed!)
      finally— always runs, even if exception was raised (cleanup)
    """
    result = {}
    try:
        data = json.loads(raw)
        value = data["score"]              # KeyError if missing
        if not isinstance(value, (int, float)):
            raise ValueError(f"score must be numeric, got {type(value).__name__}")
        result = {"score": float(value), "status": "ok"}

    except json.JSONDecodeError as e:
        result = {"score": None, "status": "parse_error", "detail": str(e)}

    except KeyError:
        result = {"score": None, "status": "missing_key"}

    except ValueError as e:
        result = {"score": None, "status": "validation_error", "detail": str(e)}

    else:
        # Only runs if no exception occurred — good place for success logging
        print(f"  [else] success — score={result['score']}")

    finally:
        # Always runs — use for cleanup (close files, release locks)
        print(f"  [finally] done processing, status={result.get('status')}")

    return result


test_inputs = [
    '{"score": 0.87}',          # valid
    '{"score": "not-a-number"}', # ValueError
    '{"other": 1}',             # KeyError
    'not json',                  # JSONDecodeError
]

for inp in test_inputs:
    print(f"Input: {inp}")
    r = process_response(inp)
    print(f"Result: {r}\n")

Input: {"score": 0.87}
  [else] success — score=0.87
  [finally] done processing, status=ok
Result: {'score': 0.87, 'status': 'ok'}

Input: {"score": "not-a-number"}
  [finally] done processing, status=validation_error
Result: {'score': None, 'status': 'validation_error', 'detail': 'score must be numeric, got str'}

Input: {"other": 1}
  [finally] done processing, status=missing_key
Result: {'score': None, 'status': 'missing_key'}

Input: not json
  [finally] done processing, status=parse_error
Result: {'score': None, 'status': 'parse_error', 'detail': 'Expecting value: line 1 column 1 (char 0)'}



In [25]:
# Re-raising exceptions — when you catch to log but still want the caller to know
def load_config(path: str) -> dict:
    try:
        return json.loads(open(path).read())
    except FileNotFoundError:
        print(f"Config file not found: {path}")
        raise    # re-raise the SAME exception — caller still gets it
    except json.JSONDecodeError as e:
        # Wrap in a more descriptive error
        raise ValueError(f"Config file {path!r} is invalid JSON: {e}") from e

try:
    load_config("missing_config.json")
except FileNotFoundError as e:
    print(f"Caller caught: {e}")

Config file not found: missing_config.json
Caller caught: [Errno 2] No such file or directory: 'missing_config.json'


---
## 4. Logging vs `print`

> **Plain English:** `print()` is you yelling at the TV during a live flight. `logging` is the flight recorder — it runs the whole time, survives the crash, and tells investigators exactly what happened at 14:37:22 UTC.

The key differences:

| `print` | `logging` |
|---|---|
| No timestamp | Timestamps on every message |
| No severity | Levels: DEBUG, INFO, WARNING, ERROR, CRITICAL |
| Always visible | Configurable — filter by level |
| Hard to disable | One line to silence all debug output |
| No context | Module name, line number, process ID available |
| Not structured | Can output JSON for log aggregation systems |

In [27]:
import logging

# Basic configuration — do this ONCE at the top of your entry point
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)-8s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

# Get a named logger — use the module/class name
log = logging.getLogger("demo")

# The five levels — in order of severity
log.debug("Fine-grained detail — only shown during debugging")   # level 10
log.info("Normal operation — something meaningful happened")      # level 20
log.warning("Something unexpected, but we can continue")         # level 30
log.error("Something failed — human attention needed")           # level 40
log.critical("System-level failure — stop everything")           # level 50

21:03:11 [DEBUG   ] demo: Fine-grained detail — only shown during debugging
21:03:11 [INFO    ] demo: Normal operation — something meaningful happened
21:03:11 [WARNING ] demo: Something unexpected, but we can continue
21:03:11 [ERROR   ] demo: Something failed — human attention needed
21:03:11 [CRITICAL] demo: System-level failure — stop everything


In [30]:
# Logging in the context of LLM testing
log = logging.getLogger("llm_test")

def run_test_case(case_id: str, prompt: str) -> dict:
    log.info("Starting test case %s", case_id)   # use % formatting, not f-strings
    # Note: use %s, %d, %f — NOT f-strings — because the message is only formatted
    # if the log level is active. f-strings format eagerly, wasting CPU.

    import random, time
    start = time.time()

    try:
        # Simulate an occasional failure
        if random.random() < 0.3:
            raise ConnectionError("Simulated network blip")
        response = f"Answer for {prompt[:20]}"
        latency  = time.time() - start
        log.info("Test %s passed in %.3fs", case_id, latency)
        return {"status": "pass", "response": response, "latency_ms": latency * 1000}

    except ConnectionError as e:
        log.warning("Test %s: connection error — %s", case_id, e)
        return {"status": "error", "detail": str(e)}

# Run a few test cases
for i in range(4):
    result = run_test_case(f"tc-{i:03d}", f"Test prompt #{i}")
    print(result)

21:04:23 [INFO    ] llm_test: Starting test case tc-000
21:04:23 [INFO    ] llm_test: Test tc-000 passed in 0.000s
21:04:23 [INFO    ] llm_test: Starting test case tc-001
21:04:23 [INFO    ] llm_test: Test tc-001 passed in 0.000s
21:04:23 [INFO    ] llm_test: Starting test case tc-002
21:04:23 [INFO    ] llm_test: Test tc-002 passed in 0.000s
21:04:23 [INFO    ] llm_test: Starting test case tc-003
21:04:23 [INFO    ] llm_test: Test tc-003 passed in 0.000s


{'status': 'pass', 'response': 'Answer for Test prompt #0', 'latency_ms': 0.0021457672119140625}
{'status': 'pass', 'response': 'Answer for Test prompt #1', 'latency_ms': 0.0021457672119140625}
{'status': 'pass', 'response': 'Answer for Test prompt #2', 'latency_ms': 0.0011920928955078125}
{'status': 'pass', 'response': 'Answer for Test prompt #3', 'latency_ms': 0.0021457672119140625}


In [31]:
# log.exception — logs ERROR + full traceback in one call (use inside except blocks)
log = logging.getLogger("exception_demo")

try:
    data = json.loads("not valid json")
except json.JSONDecodeError:
    log.exception("Failed to parse response — full traceback below")
    # This prints the ERROR message AND the traceback. Perfect for CI logs.

21:04:27 [ERROR   ] exception_demo: Failed to parse response — full traceback below
Traceback (most recent call last):
  File "/var/folders/1f/2t54lvjd7xg59dbzzkj0wr4m0000gp/T/ipykernel_78345/1868087202.py", line 5, in <module>
    data = json.loads("not valid json")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/json/decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/json/decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)


---
## 5. Log Levels in Practice

Choosing the right level is a skill. Here's a decision guide:

| Level | Use when… | Example |
|---|---|---|
| `DEBUG` | Any internal state useful for debugging | Token count, raw response before parsing |
| `INFO` | Normal milestones worth recording | Test case started, test passed, latency |
| `WARNING` | Something unexpected that doesn't stop the run | Retry triggered, response was truncated |
| `ERROR` | A test case failed due to an exception | JSONDecodeError, APIConnectionError |
| `CRITICAL` | The whole test run must abort | Auth failure, out of API credits |

In [33]:
# Controlling level — in production, set to INFO; during debugging, set to DEBUG
import logging

# Create a logger with its own handler (so we can set level independently)
logger = logging.getLogger("level_demo")
logger.setLevel(logging.WARNING)   # only WARNING and above will show

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(handler)

logger.debug("This will NOT appear (DEBUG < WARNING)")    # hidden
logger.info("This will NOT appear (INFO < WARNING)")      # hidden
logger.warning("This WILL appear")
logger.error("This WILL appear")

21:06:52 [WARNING ] level_demo: This WILL appear
ERROR: This WILL appear
ERROR: This WILL appear
21:06:52 [ERROR   ] level_demo: This WILL appear


---
## 6a. Understanding Decorators

Before we build the retry decorator, we need to understand **decorators** — because the retry pattern is *entirely* built on this concept.

> **Plain English:** a decorator is a wrapper around a function. Like a gift wrapper — the box inside (your function) is unchanged, but you've added something on the outside (logging, retries, timing, auth checks) without touching the original code.

**The fundamental idea:** in Python, functions are objects. You can pass a function to another function, and that function can return a new, wrapped version.

```
original_function
      │
      ▼
 decorator(original_function)
      │
      ▼
 wrapped_function   ← same name, extra behavior
```

In [27]:
# ── Step 1: Functions are objects — you can pass them around ────────────────

def greet(name: str) -> str:
    return f"Hello, {name}!"

def bye(name: str):
    return f"Bye, {name}"

def shout(fn):           # fn is a function passed as an argument
    result = fn("world")

    wrapped = f"***** { result } *****"
    return wrapped

print(shout(greet))       # "HELLO, WORLD!"
print(shout(bye))
# print(type(greet))        # <class 'function'>

***** Hello, world! *****
***** Bye, world *****


In [21]:
def sum_of_numbers(n1: int,n2: int):
    return n1+n2

def sub_of_numbers(n1: int,n2: int):
    return n1-n2


def square(fn):
    value = fn(2,3)
    return value*value

print(square(sum_of_numbers))
print(square(sub_of_numbers))

25
1


In [ ]:
# ── Step 2: Write a decorator by hand ───────────────────────────────────────
# A decorator is a function that:
#   1. Takes a function as input
#   2. Defines a new "wrapper" function inside
#   3. Returns the wrapper

def timer_decorator(fn):
    """Wraps any function and prints how long it took."""
    def wrapper(*args, **kwargs):           # *args/**kwargs = accept any arguments
        import time
        start = time.time()
        result = fn(*args, **kwargs)
        end = time.time()       # call the original function
        elapsed = end - start
        print(f"  [{fn.__name__}] took {elapsed*1000:.1f}ms")
        return result                       # return the original result unchanged
    return wrapper                          # return the wrapper, not the result


# Manual way — call the decorator explicitly
def add(a, b,c):
    return a + b

def sub(a,b):
    return a-b

timed_add = timer_decorator(add)  
timed_sub = timer_decorator(sub)  

timed_add(2,3)     # 7, plus timing output
timed_sub(5,2)

  [add] took 0.0ms
  [sub] took 0.0ms


3

In [32]:
@timer_decorator
def add_nums(a,b):
    return a+b

add_nums(1,2)

  [add_nums] took 0.0ms


3

In [34]:
# ── Step 3: The @ syntax — syntactic sugar ───────────────────────────────────
# @timer_decorator above a def is EXACTLY equivalent to:
#     my_fn = timer_decorator(my_fn)
# It's just shorter and sits right on top of the function definition.

@timer_decorator
def multiply(a, b):
    return a * b

print(multiply(6, 7))    # 42, with timing

# Confirm the decorator replaced the function
print(multiply.__name__)  # prints "wrapper" — the original name is LOST
                          # this is the problem functools.wraps fixes

  [multiply] took 0.0ms
42
wrapper


In [35]:
# ── Step 4: functools.wraps — preserve the original function's identity ──────
# Without @wraps, the wrapped function loses its __name__ and __doc__.
# Debugging, logging, and pytest all rely on these. Always use @wraps.

from functools import wraps

def better_timer(fn):
    @wraps(fn)           # copies __name__, __doc__, __module__ from fn → wrapper
    def wrapper(*args, **kwargs):
        import time
        start = time.time()
        result = fn(*args, **kwargs)
        elapsed = time.time() - start
        print(f"  [{fn.__name__}] took {elapsed*1000:.1f}ms")
        return result
    return wrapper

@better_timer
def divide(a, b):
    """Divide a by b."""
    return a / b

print(divide(10, 2))
print(divide.__name__)   # "divide" — preserved ✓
print(divide.__doc__)    # "Divide a by b." — preserved ✓

  [divide] took 0.0ms
5.0
divide
Divide a by b.


In [ ]:
# ── Step 5: Decorator factory — a decorator that takes its own arguments ─────
# Our retry decorator needs parameters: attempts=3, base_delay=1.0
# To pass arguments TO the decorator, we need one more layer of nesting.
# The outer function receives the config, returns the real decorator.

#  retry(attempts=3)   ← this CALL returns the decorator
#         │
#         ▼
#   decorator(fn)      ← this receives your function
#         │
#         ▼
#     wrapper(...)     ← this runs every time you call your function


def configurable_timer(unit: str = "ms"):
    """Decorator factory: takes config, returns a decorator."""
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            import time
            start = time.time()
            result = fn(*args, **kwargs)
            elapsed = time.time() - start
            value = elapsed * 1000 if unit == "ms" else elapsed
            print(f"  [{fn.__name__}] {value:.2f}{unit}")
            return result
        return wrapper
    return decorator   # ← return the decorator, not wrapper

# @configurable_timer(unit="ms")
# def fast_fn():
#     return sum(range(10_000))

# @configurable_timer(unit="s")
# def slow_fn():
#     import time; time.sleep(0.05)

# fast_fn()
# slow_fn()
def parameterized_timer(add_s = 2):
    def timer_decorator(fn):
        """Wraps any function and prints how long it took."""
        def wrapper(*args, **kwargs):           # *args/**kwargs = accept any arguments
            import time
            start = time.time()
            result = fn(*args, **kwargs)
            end = time.time()       # call the original function
            elapsed = end - start + add_s
            print(f"[{fn.__name__}] took {elapsed*1000:.1f}ms")
            return result                       # return the original result unchanged
        return wrapper   
    return timer_decorator


@parameterized_timer(3)
def sum_of_numbers(a,b):
    return a+b

sum_of_numbers(1,2)
# 🔧 Try it: add a @configurable_timer(unit="ms") to one of your own functions

[sum_of_numbers] took 3000.0ms


3

In [51]:
def parametered_ji(num):
    def Ji_decorator(fn):
        def wrapper(*args,**kwargs):
            greeting = fn(*args,**kwargs)
            greeting_with_ji = greeting + " Ji"*num
            return greeting_with_ji
        return wrapper
    return Ji_decorator

@parametered_ji(2)
def Hello(name):
    greeting = " Hello " + name
    return greeting

@parametered_ji(3)
def Namaste(name):
    greeting = " Namaste " + name
    return greeting

@parametered_ji(4)
def Namaskara(name):
    greeting = " Namaskara " + name
    return greeting


# Namaskara("Jitesh")
Namaste("Jitesh")

' Namaste Jitesh Ji Ji Ji'

---
## 6b. Understanding Generators

A **generator** is a function that produces values one at a time using the `yield` keyword, pausing between each one. It remembers where it left off.

> **Plain English:** a generator is like a waiter who brings dishes one at a time. A regular function is like a waiter who goes to the kitchen, assembles the entire meal, carries everything out at once, and sets it all on the table. A generator waiter brings the soup, comes back when you're ready, brings the salad, comes back, brings the main course. You don't need to keep everything on the table at once.

**Why this matters here:**  
- The retry loop in our decorator is essentially a generator of attempts.
- `pytest` uses generators internally for fixture `yield` teardown.
- Streaming LLM responses are generators — chunks arrive one by one.

**Key difference from a regular function:**

| Regular function | Generator function |
|---|---|
| Runs to completion, returns once | Pauses at each `yield`, resumes on `next()` |
| Returns a value | Yields a sequence of values |
| All values computed immediately | Values computed on demand (lazy) |
| `return result` | `yield result` |

In [8]:
# ── Generator basics — yield pauses the function ────────────────────────────

def count_up(start: int, stop: int):
    """A generator that yields integers from start to stop."""
    print(f"  Generator starting at {start}")
    n = start
    while n <= stop:
        print(f"  About to yield {n}")
        yield n                  # ← pause here, hand n to the caller
        print(f"  Resumed after yielding {n}")
        n += 1
    print("  Generator exhausted")


# Using a generator in a for loop — next() is called automatically each iteration
print("=== for loop ===")
for value in count_up(1, 3):
    print(f"  Caller received: {value}")

print()

# Using next() manually to step one at a time
print("=== manual next() ===")
gen = count_up(10, 12)
print(f"Got: {next(gen)}")    # runs until first yield, pauses
print(f"Got: {next(gen)}")    # resumes, runs until second yield, pauses
print(f"Got: {next(gen)}")    # resumes, runs until third yield, pauses
# next(gen) here would raise StopIteration — generator is exhausted

=== for loop ===
  Generator starting at 1
  About to yield 1
  Caller received: 1
  Resumed after yielding 1
  About to yield 2
  Caller received: 2
  Resumed after yielding 2
  About to yield 3
  Caller received: 3
  Resumed after yielding 3
  Generator exhausted

=== manual next() ===
  Generator starting at 10
  About to yield 10
Got: 10
  Resumed after yielding 10
  About to yield 11
Got: 11
  Resumed after yielding 11
  About to yield 12
Got: 12


In [ ]:
# ── Generators are lazy — memory comparison ─────────────────────────────────
import sys

# List comprehension — computes and stores all values immediately
big_list = [x * 2 for x in range(1_000_000)]
print(f"List (1M items) in memory:  {sys.getsizeof(big_list):>10,} bytes")

# Generator expression — stores only the recipe, not the values
big_gen = (x * 2 for x in range(1_000_000))   # () not []
print(f"Generator (1M items):       {sys.getsizeof(big_gen):>10,} bytes")

# Both produce the same values — generator is just lazier
print(f"\nFirst 5 from list: {big_list[:5]}")
print(f"First 5 from gen:  {[next(big_gen) for _ in range(5)]}")

print("\nRule: if you don't need ALL values at once, use a generator expression.")

In [ ]:
# ── Generators in AI testing: streaming LLM responses ───────────────────────
# The OpenAI SDK streaming API returns a generator — chunks arrive one by one.
# You iterate it with a for loop; each iteration calls next() automatically.

def simulate_streaming_response(text: str):
    """Simulates how a streaming LLM SDK works internally."""
    words = text.split()
    for word in words:
        import time
        time.sleep(0.02)      # simulate network delay per token
        yield word + " "      # yield one token at a time

# This is exactly how you'd handle streaming in production:
print("Streaming output: ", end="", flush=True)
for token in simulate_streaming_response("The capital of France is Paris"):
    print(token, end="", flush=True)
print()  # newline at end

# With the real SDK it would be:
# for chunk in client.chat.completions.create(..., stream=True):
#     if chunk.choices[0].delta.content:
#         print(chunk.choices[0].delta.content, end="", flush=True)
print("\n(Real SDK streaming works the same way — it yields chunks from the network)")

# 🔧 Try it: modify simulate_streaming_response to uppercase every 3rd word

---
## 6c. Retries with Exponential Backoff

Now we combine **decorators** (6a) and the **looping/iteration** pattern (6b) into one production tool.

LLM APIs fail transiently — rate limits (429), upstream 500s, network resets. A test framework that gives up on the first failure is useless in CI. We want: try → fail → wait → try again → wait longer → try again → give up.

> **Plain English:** exponential backoff = progressively longer waits. First knock — wait 1s. No answer? Wait 2s. Still nothing? Wait 4s. After N tries, accept it's broken and raise. The increasing delays prevent hammering an already-stressed server.

**Why jitter?** If 100 test workers all fail at the same moment and all retry after exactly 1s, they'll all hit the server simultaneously again. Adding `+ random.random()` spreads retries across a window — this is called the "thundering herd" problem.

**The delay formula:**

```
attempt 1 fails → delay = 1.0 * 2^0 + jitter = 1.0 + 0.x  ≈ 1–2s
attempt 2 fails → delay = 1.0 * 2^1 + jitter = 2.0 + 0.x  ≈ 2–3s
attempt 3 fails → delay = 1.0 * 2^2 + jitter = 4.0 + 0.x  ≈ 4–5s
attempt 4 fails → give up, re-raise the last exception
```

**Which errors to retry vs. not:**

| Exception | Retry? | Reason |
|---|---|---|
| `RateLimitError` (429) | ✅ Yes | Temporary — will clear after delay |
| `APIConnectionError` | ✅ Yes | Network blip — usually self-heals |
| `APIError` (500/503) | ✅ Yes | Provider outage — often brief |
| `AuthenticationError` | ❌ No | Wrong key — retrying won't fix it |
| `ValueError` | ❌ No | Your code is wrong — fix the code |
| `json.JSONDecodeError` | ❌ No | Bad response shape — retry may repeat it |

In [ ]:
import time
import random
import logging
from functools import wraps
from typing import Type

# ── The retry decorator — annotated line by line ─────────────────────────────
#
# Structure recap (from 6a):
#   retry(attempts, base_delay, exceptions)   ← FACTORY: returns the decorator
#     └─ decorator(fn)                         ← DECORATOR: wraps the function
#           └─ wrapper(*args, **kwargs)         ← WRAPPER: runs on every call

def retry(
    attempts: int = 3,
    base_delay: float = 1.0,
    exceptions: tuple[Type[Exception], ...] = (Exception,),
):
    """
    Decorator factory. Usage:
        @retry(attempts=3, base_delay=1.0, exceptions=(RateLimitError, NetworkError))
        def call_api(prompt): ...
    """
    def decorator(fn):
        @wraps(fn)                                  # preserve fn.__name__, fn.__doc__
        def wrapper(*args, **kwargs):
            log = logging.getLogger(f"retry.{fn.__name__}")

            for attempt in range(1, attempts + 1): # attempt: 1, 2, 3, ...
                try:
                    return fn(*args, **kwargs)      # 1. try the real function

                except exceptions as exc:           # 2. caught ONLY the listed types
                    if attempt == attempts:
                        # last attempt exhausted — log final failure and re-raise
                        log.error(
                            "[%s] All %d attempts failed. Last error: %s",
                            fn.__name__, attempts, exc
                        )
                        raise                       # re-raise original exception

                    # 3. compute delay: base * 2^(attempt-1) + random jitter
                    delay = base_delay * (2 ** (attempt - 1)) + random.random()
                    log.warning(
                        "[%s] Attempt %d/%d failed — %s: %s. Retrying in %.2fs ...",
                        fn.__name__, attempt, attempts,
                        type(exc).__name__, exc, delay
                    )
                    time.sleep(delay)               # 4. wait before next attempt

            # wrapper returns here only if fn() succeeded inside the loop
        return wrapper
    return decorator


# ── Visualize the delay progression ──────────────────────────────────────────
print("Delay schedule for retry(attempts=5, base_delay=1.0):\n")
print(f"  {'Attempt':<10} {'Formula':<30} {'Min delay':>10}  {'Max delay':>10}")
print("  " + "-" * 64)
for a in range(1, 5):   # attempts 1–4 (attempt 5 would give up)
    min_d = 1.0 * (2 ** (a - 1))
    max_d = min_d + 1.0
    formula = f"1.0 × 2^{a-1} + jitter"
    print(f"  {a:<10} {formula:<30} {min_d:>8.1f}s   {max_d:>8.1f}s")
print(f"\n  Total max wait (4 delays): ~{sum(1.0*(2**(a-1))+1 for a in range(1,5)):.0f}s before giving up")

In [10]:
# Demonstrate the retry decorator
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S", force=True)

call_count = 0

@retry(attempts=4, base_delay=0.1)   # short delay for demo
def flaky_api_call(prompt: str) -> str:
    """Simulates an API that fails 70% of the time."""
    global call_count
    call_count += 1
    if random.random() < 0.7:
        raise ConnectionError(f"Network blip on attempt {call_count}")
    return f"Response to: {prompt}"

call_count = 0
try:
    result = flaky_api_call("What is AI?")
    print(f"\nSuccess: {result}")
    print(f"Total API calls made: {call_count}")
except ConnectionError:
    print(f"\nFailed after {call_count} attempts")

19:56:17 [WARNING] Attempt 1/4 failed (ConnectionError: Network blip on attempt 1) — retrying in 0.8s
19:56:17 [WARNING] Attempt 2/4 failed (ConnectionError: Network blip on attempt 2) — retrying in 0.5s
19:56:18 [WARNING] Attempt 3/4 failed (ConnectionError: Network blip on attempt 3) — retrying in 1.4s
19:56:19 [ERROR] Giving up after 4 attempts: Network blip on attempt 4



Failed after 4 attempts


In [11]:
# Retrying only specific exceptions — important for auth errors
# You should NEVER retry an AuthenticationError — it won't fix itself.

class AuthenticationError(Exception): pass
class RateLimitError(Exception): pass
class NetworkError(Exception): pass

# Only retry on transient errors
@retry(attempts=3, base_delay=0.1, exceptions=(RateLimitError, NetworkError))
def smart_api_call(fail_type: str = "network") -> str:
    if fail_type == "auth":
        raise AuthenticationError("Invalid API key")   # NOT retried
    if fail_type == "rate":
        raise RateLimitError("429 Too Many Requests")  # retried
    if fail_type == "network":
        raise NetworkError("Connection reset")         # retried
    return "success"

# Auth error — propagates immediately (no retry)
try:
    smart_api_call("auth")
except AuthenticationError as e:
    print(f"Auth error (not retried): {e}")

# Rate limit — retried 3 times
try:
    smart_api_call("rate")
except RateLimitError as e:
    print(f"Rate limit (gave up after retries): {e}")

19:56:23 [WARNING] Attempt 1/3 failed (RateLimitError: 429 Too Many Requests) — retrying in 0.5s


Auth error (not retried): Invalid API key


19:56:23 [WARNING] Attempt 2/3 failed (RateLimitError: 429 Too Many Requests) — retrying in 0.6s
19:56:24 [ERROR] Giving up after 3 attempts: 429 Too Many Requests


Rate limit (gave up after retries): 429 Too Many Requests


In [ ]:
# ── Connecting the dots: decorator + generator pattern ───────────────────────
# The retry decorator IS a generator of attempts — just expressed as a loop.
# Here's the same logic rewritten as an explicit generator to show the parallel:

def attempt_sequence(attempts: int, base_delay: float):
    """Generator that yields (attempt_number, delay_after_this_attempt)."""
    for attempt in range(1, attempts + 1):
        is_last = (attempt == attempts)
        delay = base_delay * (2 ** (attempt - 1)) + random.random() if not is_last else 0
        yield attempt, delay, is_last


def retry_with_generator(fn, attempts=3, base_delay=0.1):
    """Same retry logic, but using an explicit generator to produce attempts."""
    for attempt, delay, is_last in attempt_sequence(attempts, base_delay):
        try:
            return fn()
        except Exception as exc:
            if is_last:
                print(f"  Gave up after {attempt} attempts: {exc}")
                raise
            print(f"  Attempt {attempt}/{attempts} failed ({exc}) — waiting {delay:.2f}s")
            time.sleep(delay)


# Test it
call_n = 0
def sometimes_fails():
    global call_n
    call_n += 1
    if call_n < 3:
        raise ConnectionError(f"fail #{call_n}")
    return f"success on call #{call_n}"

call_n = 0
result = retry_with_generator(sometimes_fails, attempts=4)
print(f"\nResult: {result}")

# 🔧 Try it: change the failure condition so it never succeeds — observe the give-up log

---
## 7. Putting It Together — A Production-Grade LLM Client

Combining exceptions + logging + retries into a single reusable client. This is the pattern used in real AI testing frameworks.

In [12]:
import os
import logging
import time
import random
from functools import wraps
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)-8s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("llm_client")

PROVIDER       = os.getenv("PROVIDER", "ollama").lower()
MODEL          = os.getenv("DEMO_MODEL", "llama3.2:3b")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")


# ── Retry decorator (same as above) ─────────────────────────────────────────
def retry(attempts=3, base_delay=1.0):
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as exc:
                    if attempt == attempts:
                        log.error("Giving up after %d attempts: %s", attempt, exc)
                        raise
                    delay = base_delay * (2 ** (attempt - 1)) + random.random()
                    log.warning("Attempt %d/%d failed (%s) — retry in %.1fs",
                                attempt, attempts, type(exc).__name__, delay)
                    time.sleep(delay)
        return wrapper
    return decorator


# ── The client ──────────────────────────────────────────────────────────────
@retry(attempts=3, base_delay=0.5)
def generate(prompt: str, temperature: float = 0.3, max_tokens: int = 500) -> str:
    """
    Call the configured LLM provider and return the text response.
    Retries up to 3 times on any transient error.
    """
    from openai import OpenAI, RateLimitError, APIConnectionError, APIError

    if PROVIDER == "openai":
        client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    else:
        client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

    log.info("generate: provider=%s model=%s prompt_len=%d", PROVIDER, MODEL, len(prompt))

    try:
        start = time.time()
        resp  = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=max_tokens,
            timeout=30,
        )
        latency = time.time() - start
        content = resp.choices[0].message.content.strip()
        log.info("generate: OK latency=%.2fs tokens=%d",
                 latency, resp.usage.total_tokens if resp.usage else -1)
        return content

    except RateLimitError:
        log.warning("Rate limited by provider")
        raise
    except APIConnectionError as exc:
        log.warning("Network/connection error: %s", exc)
        raise
    except APIError as exc:
        log.error("Provider API error: %s", exc)
        raise


print("Client defined. Run the next cell to make a real call.")

Client defined. Run the next cell to make a real call.


In [13]:
# Make real calls — requires Ollama running or PROVIDER=openai
prompts = [
    "What is the capital of Japan?",
    "Write a one-sentence tagline for an AI testing tool.",
    "What does 'pytest fixture' mean? One sentence.",
]

for i, p in enumerate(prompts, 1):
    log.info("=== Prompt %d/%d ===", i, len(prompts))
    try:
        answer = generate(p)
        log.info("Answer: %s", answer)
        print(f"\n[{i}] {p}\n    → {answer}")
    except Exception as exc:
        log.error("Prompt %d failed: %s", i, exc)
        print(f"\n[{i}] FAILED: {exc}")

19:56:41 [INFO    ] llm_client: === Prompt 1/3 ===
19:56:41 [INFO    ] llm_client: generate: provider=ollama model=llama3.2:3b prompt_len=29
19:56:50 [INFO    ] httpx: HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
19:56:50 [INFO    ] llm_client: generate: OK latency=9.11s tokens=40
19:56:50 [INFO    ] llm_client: Answer: The capital of Japan is Tokyo.
19:56:50 [INFO    ] llm_client: === Prompt 2/3 ===
19:56:50 [INFO    ] llm_client: generate: provider=ollama model=llama3.2:3b prompt_len=52



[1] What is the capital of Japan?
    → The capital of Japan is Tokyo.


19:56:52 [INFO    ] httpx: HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
19:56:52 [INFO    ] llm_client: generate: OK latency=1.53s tokens=60
19:56:52 [INFO    ] llm_client: Answer: "Test, refine, and unleash the full potential of your AI with our intuitive and comprehensive testing platform."
19:56:52 [INFO    ] llm_client: === Prompt 3/3 ===
19:56:52 [INFO    ] llm_client: generate: provider=ollama model=llama3.2:3b prompt_len=46



[2] Write a one-sentence tagline for an AI testing tool.
    → "Test, refine, and unleash the full potential of your AI with our intuitive and comprehensive testing platform."


19:56:54 [INFO    ] httpx: HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
19:56:54 [INFO    ] llm_client: generate: OK latency=2.09s tokens=74
19:56:54 [INFO    ] llm_client: Answer: A pytest fixture is a reusable block of code that provides a fixed baseline so that tests can be run multiple times without having to recreate the same environment, making testing more efficient and reliable.



[3] What does 'pytest fixture' mean? One sentence.
    → A pytest fixture is a reusable block of code that provides a fixed baseline so that tests can be run multiple times without having to recreate the same environment, making testing more efficient and reliable.


In [14]:
import logging

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s [%(levelname)-8s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True) # override any previous logging configuration)

logger1 = logging.getLogger('example_function')
logger2 = logging.getLogger('implementation_function')

In [15]:
def example_function():
    logger1.info("The Function execution started")
    logger1.warning("This is a warning in example_function")
    logger1.debug("This is a debug message in example_function")


def implementation_function():
    logger2.info("The implementation is running")
    logger2.warning("This is a warning in implementation_function")




In [16]:
example_function()

19:56:54 [WARNING ] example_function: This is a warning in example_function


In [17]:
implementation_function()

19:56:54 [WARNING ] implementation_function: This is a warning in implementation_function


In [18]:
# 🔧 Try it: intentionally break the client and observe retry behavior
# Option A: change MODEL to a name that doesn't exist (e.g., "llama999:9b")
#            → you'll see 3 retry attempts before it gives up
# Option B: set OLLAMA_BASE_URL to a port nothing is listening on
#            → APIConnectionError, then retries
#
# After you observe the retries, fix it back and run the prompts again.
# The goal: understand what a retry log looks like — you'll see this in CI.

---
## Day 3 Summary

| Concept | Key syntax | Coming back in |
|---|---|---|
| Exceptions | `try / except SpecificError as e:` | Day 4 (API errors), Day 5 (test failures) |
| Exception hierarchy | Know which class to catch | Day 4, 6 |
| `else` / `finally` | Only-on-success / always-runs | Day 4 cleanup patterns |
| Logging | `log.info("msg %s", val)` | Every day from here |
| Log levels | DEBUG/INFO/WARNING/ERROR | Day 6 (framework), Day 7 (CI output) |
| **Decorator** | `def decorator(fn): ... return wrapper` | Day 5 (pytest fixtures), Day 6 (framework) |
| **`@wraps(fn)`** | Preserve `__name__` / `__doc__` | Every decorator you write |
| **Decorator factory** | Three-level nesting: factory → decorator → wrapper | `@retry(attempts=3)` |
| **Generator** | `yield value` — pause and resume | Day 5 (`yield` fixtures), streaming APIs |
| **Generator expression** | `(x for x in y)` vs `[x for x in y]` | Memory-efficient iteration |
| Retry decorator | `@retry(attempts=3, exceptions=(...))` | Day 4 (client), Module 4+ |
| Exponential backoff | `delay = base * 2^(attempt-1) + jitter` | All LLM client code |
| Thundering herd | Add `+ random.random()` to delay | Any distributed retry logic |

**The chain:** `logging` tells you what happened → `retry` keeps you running → `decorator` makes retry reusable → `generator` is the mental model underneath both `yield` fixtures and streaming.

**Exercise:** [`exercises/day3_exercise.md`](../exercises/day3_exercise.md)  
**Next:** Day 4 — API testing with `requests`, SDKs, and `.env` secrets